# Chapter 4 Computational Lab
## Conditional Probability and Independence

This notebook accompanies Chapter 4 of *Probability Theory with Python and AI*.

The chapter begins with a simple idea: **new information changes probabilities**. Conditional probability formalizes this update. Independence is the exceptional situation in which learning one event does not change the probability of another.

### Learning goals

By the end of this lab you should be able to:

1. compute and interpret conditional probabilities;
2. verify that conditioning produces a new probability measure;
3. use multiplication and chain rules;
4. use partitions and the law of total probability;
5. distinguish prior, likelihood, evidence and posterior in Bayes' theorem;
6. analyze the Monty Hall problem under an explicitly stated host protocol;
7. distinguish disjointness from independence;
8. test independence using both product and conditional criteria;
9. distinguish pairwise from mutual independence;
10. use the complement-pattern characterization of mutual independence;
11. compare the exact independent-event union formula with Boole's inequality;
12. understand conditional independence and common latent causes;
13. audit AI-generated conditional-probability arguments.

> **Core modelling rule.** A conditional probability is never just a formula. The event after the vertical bar specifies the information that has become known.


## 0. Setup

The notebook uses exact rational arithmetic for finite examples and simulation only when simulation genuinely adds insight.


In [ ]:
from fractions import Fraction
from itertools import product, combinations
from random import Random
import math

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def fmt_fraction(x):
    x = Fraction(x)
    if x.denominator == 1:
        return str(x.numerator)
    return rf"\frac{{{x.numerator}}}{{{x.denominator}}}"


def fmt_set(A):
    if not A:
        return r"\varnothing"
    return r"\{" + ",".join(map(str, sorted(A))) + r"\}"


def parse_set(text):
    text = text.strip()
    if not text:
        return frozenset()
    return frozenset(int(x.strip()) for x in text.split(",") if x.strip())


def power_set(items):
    items = tuple(items)
    return {
        frozenset(c)
        for r in range(len(items) + 1)
        for c in combinations(items, r)
    }


def probability(event, mass):
    event = frozenset(event)
    omega = frozenset(mass)
    if not event <= omega:
        raise ValueError("Event contains outcomes outside Ω.")
    return sum((mass[w] for w in event), Fraction(0, 1))


def conditional_probability(A, B, mass):
    PB = probability(B, mass)
    if PB == 0:
        raise ZeroDivisionError("Conditional probability requires P(B)>0.")
    return probability(frozenset(A) & frozenset(B), mass) / PB


def is_independent(A, B, mass):
    return (
        probability(frozenset(A) & frozenset(B), mass)
        == probability(A, mass) * probability(B, mass)
    )


def show_result(title, *latex_lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'>"
        f"<b>{title}</b></div>"
    ))
    for line in latex_lines:
        display(Math(line))
    if note:
        display(Markdown(note))


display(HTML(
    "<div style='padding:10px;border:1px solid'>"
    "<b>Setup complete.</b> Conditional-probability tools are ready."
    "</div>"
))


## 1. Conditioning on new information

For events $A,B\in\mathcal F$ with $P(B)>0$,

$$
P(A\mid B)
=
\frac{P(A\cap B)}{P(B)}.
$$

Once $B$ is known to have occurred:

- outcomes outside $B$ are discarded;
- the remaining probability mass is renormalized so that $B$ has conditional probability $1$.

The order matters: in general,

$$
P(A\mid B)\ne P(B\mid A).
$$


In [ ]:
die_mass = {i: Fraction(1, 6) for i in range(1, 7)}
die_omega = frozenset(die_mass)

cond_A = widgets.Text(value="2,4,6", description="A")
cond_B = widgets.Text(value="1,2,3,4", description="B")
cond_output = widgets.Output()


def update_conditioning(*_):
    with cond_output:
        clear_output(wait=True)

        try:
            A = parse_set(cond_A.value)
            B = parse_set(cond_B.value)
        except ValueError:
            display(Markdown("**Use comma-separated integers.**"))
            return

        if not A <= die_omega or not B <= die_omega:
            display(Markdown("**A and B must be subsets of Ω={1,...,6}.**"))
            return

        PB = probability(B, die_mass)
        if PB == 0:
            display(Markdown("**Conditional probability is undefined because P(B)=0.**"))
            return

        PAB = probability(A & B, die_mass)
        PAgivenB = conditional_probability(A, B, die_mass)

        display(Math(r"A=" + fmt_set(A)))
        display(Math(r"B=" + fmt_set(B)))
        display(Math(r"A\cap B=" + fmt_set(A & B)))
        display(Math(
            r"P(A\mid B)="
            + r"\frac{" + fmt_fraction(PAB) + r"}{" + fmt_fraction(PB) + r"}"
            + "=" + fmt_fraction(PAgivenB)
        ))

        PA = probability(A, die_mass)
        if PA > 0:
            PBgivenA = conditional_probability(B, A, die_mass)
            display(Math(
                r"P(B\mid A)=" + fmt_fraction(PBgivenA)
            ))


for control in (cond_A, cond_B):
    control.observe(update_conditioning, names="value")

display(widgets.VBox([
    widgets.HBox([cond_A, cond_B]),
    cond_output,
]))
update_conditioning()


### Nested events

If $E\subseteq T$ and $P(T)>0$, then

$$
P(E\mid T)
=
\frac{P(E)}{P(T)}.
$$

But if $P(E)>0$,

$$
P(T\mid E)=1.
$$

So even for nested events, reversing the conditioning changes the question.


In [ ]:
nested_PT = widgets.FloatSlider(value=0.20, min=0.05, max=1.00, step=0.05, description="P(T)")
nested_PE = widgets.FloatSlider(value=0.05, min=0.00, max=1.00, step=0.01, description="P(E)")
nested_output = widgets.Output()


def update_nested(*_):
    with nested_output:
        clear_output(wait=True)

        PT = nested_PT.value
        PE = nested_PE.value

        if PE > PT:
            display(Markdown("**If E⊆T, then P(E) cannot exceed P(T).**"))
            return

        display(Math(
            rf"P(E\mid T)=\frac{{{PE:.2f}}}{{{PT:.2f}}}={PE/PT:.4f}"
        ))

        if PE > 0:
            display(Math(r"P(T\mid E)=1"))


for control in (nested_PT, nested_PE):
    control.observe(update_nested, names="value")

display(widgets.VBox([
    widgets.HBox([nested_PT, nested_PE]),
    nested_output,
]))
update_nested()


## 2. Conditioning produces a new probability measure

Fix $B$ with $P(B)>0$ and define

$$
P_B(A)=P(A\mid B).
$$

Then $P_B$ is itself a probability measure.

So once the conditioning information is fixed, all ordinary probability identities can be reused conditionally.


### Conditional law of a fair die

Condition on

$$
B=\{2,4,6\}.
$$

The conditional distribution is concentrated on the even outcomes:

$$
P_B(\{2\})
=
P_B(\{4\})
=
P_B(\{6\})
=
\frac13,
$$

and

$$
P_B(B^c)=0.
$$


In [ ]:
B_even = frozenset({2, 4, 6})

conditional_masses = {
    w: conditional_probability({w}, B_even, die_mass)
    for w in die_omega
}

for w in sorted(conditional_masses):
    display(Math(
        rf"P_B(\{{{w}\}})=" + fmt_fraction(conditional_masses[w])
    ))

assert sum(conditional_masses.values(), Fraction(0, 1)) == 1
display(Markdown("The conditional singleton masses sum to **1** exactly."))


## 3. Multiplication rule

Rearranging the definition gives

$$
P(A\cap B)
=
P(B)P(A\mid B),
$$

whenever $P(B)>0$.

Equivalently, if $P(A)>0$,

$$
P(A\cap B)
=
P(A)P(B\mid A).
$$

This form is especially useful for sequential experiments.


### Two draws without replacement

A bag contains three red and two blue balls. Two balls are drawn uniformly without replacement.

Let:

- $R_1$: first ball is red;
- $B_2$: second ball is blue.

Then

$$
P(R_1)=\frac35,
\qquad
P(B_2\mid R_1)=\frac24,
$$

so

$$
P(R_1\cap B_2)
=
\frac35\cdot\frac24
=
\frac3{10}.
$$


In [ ]:
draws = []
balls = ("R1", "R2", "R3", "B1", "B2")

for first in balls:
    for second in balls:
        if first != second:
            draws.append((first, second))

event_R1_B2 = [
    pair
    for pair in draws
    if pair[0].startswith("R") and pair[1].startswith("B")
]

display(Markdown(f"Total ordered draws without replacement: **{len(draws)}**"))
display(Markdown(f"Favourable ordered draws: **{len(event_R1_B2)}**"))
display(Math(
    rf"P(R_1\cap B_2)=\frac{{{len(event_R1_B2)}}}{{{len(draws)}}}"
    r"=\frac3{10}"
))


## 4. Chain rule

For events $A_1,\ldots,A_n$ with the required conditioning probabilities defined,

$$
P\left(\bigcap_{k=1}^{n}A_k\right)
=
P(A_1)
\prod_{k=2}^{n}
P\left(
A_k
\,\middle|\,
\bigcap_{j=1}^{k-1}A_j
\right).
$$

The chain rule converts a long intersection into a sequence of local conditional probabilities.


In [ ]:
chain_p1 = widgets.FloatSlider(value=0.10, min=0.01, max=1.0, step=0.01, description="P(A₁)")
chain_p2 = widgets.FloatSlider(value=0.15, min=0.01, max=1.0, step=0.01, description="P(A₂|A₁)")
chain_p3 = widgets.FloatSlider(value=0.20, min=0.01, max=1.0, step=0.01, description="P(A₃|A₁∩A₂)")
chain_output = widgets.Output()


def update_chain(*_):
    with chain_output:
        clear_output(wait=True)

        value = chain_p1.value * chain_p2.value * chain_p3.value

        display(Math(
            rf"P(A_1\cap A_2\cap A_3)"
            rf"=({chain_p1.value:.2f})"
            rf"({chain_p2.value:.2f})"
            rf"({chain_p3.value:.2f})"
            rf"={value:.6f}"
        ))


for control in (chain_p1, chain_p2, chain_p3):
    control.observe(update_chain, names="value")

display(widgets.VBox([
    chain_p1,
    chain_p2,
    chain_p3,
    chain_output,
]))
update_chain()


## 5. Partitions and the law of total probability

Events $B_1,\ldots,B_m$ form a finite partition if they are pairwise disjoint and

$$
\bigcup_{i=1}^{m}B_i=\Omega.
$$

Then

$$
P(A)
=
\sum_{i=1}^{m}
P(A\mid B_i)P(B_i).
$$

A partition splits uncertainty into mutually exclusive and exhaustive cases.


### Defects from production lines

Suppose two lines produce $60\%$ and $40\%$ of all items, with defect rates $1\%$ and $3\%$.

Then

$$
P(D)
=
(0.01)(0.60)+(0.03)(0.40)
=
0.018.
$$


In [ ]:
line1_share = widgets.FloatSlider(value=0.60, min=0.0, max=1.0, step=0.01, description="P(L₁)")
line1_defect = widgets.FloatSlider(value=0.01, min=0.0, max=0.20, step=0.005, description="P(D|L₁)")
line2_defect = widgets.FloatSlider(value=0.03, min=0.0, max=0.20, step=0.005, description="P(D|L₂)")
total_output = widgets.Output()


def update_total_probability(*_):
    with total_output:
        clear_output(wait=True)

        p1 = line1_share.value
        p2 = 1 - p1
        d1 = line1_defect.value
        d2 = line2_defect.value

        total = d1 * p1 + d2 * p2

        display(Math(
            rf"P(D)=({d1:.3f})({p1:.2f})+({d2:.3f})({p2:.2f})"
            rf"={total:.5f}"
        ))


for control in (line1_share, line1_defect, line2_defect):
    control.observe(update_total_probability, names="value")

display(widgets.VBox([
    line1_share,
    widgets.HBox([line1_defect, line2_defect]),
    total_output,
]))
update_total_probability()


## 6. Bayes' theorem

Let $B_1,\ldots,B_m$ form a partition and let $A$ be observed. Then

$$
P(B_j\mid A)
=
\frac{
P(A\mid B_j)P(B_j)
}{
\sum_{i=1}^{m}P(A\mid B_i)P(B_i)
}.
$$

Interpretation:

- $P(B_j)$: **prior**;
- $P(A\mid B_j)$: **likelihood**;
- $P(A)$: marginal probability of the evidence;
- $P(B_j\mid A)$: **posterior**.

Likelihoods across causes do not have to sum to one.


### Screening example

Suppose:

$$
P(F)=0.02,
$$

$$
P(G\mid F)=0.90,
$$

$$
P(G\mid F^c)=0.05.
$$

Then

$$
P(G)
=
(0.90)(0.02)+(0.05)(0.98)
=
0.067,
$$

and

$$
P(F\mid G)
=
\frac{(0.90)(0.02)}{0.067}
=
\frac{18}{67}
\approx0.2687.
$$


In [ ]:
bayes_prior = widgets.FloatSlider(value=0.02, min=0.001, max=0.50, step=0.001, description="prior")
bayes_sens = widgets.FloatSlider(value=0.90, min=0.01, max=1.00, step=0.01, description="sensitivity")
bayes_fp = widgets.FloatSlider(value=0.05, min=0.00, max=0.50, step=0.01, description="false positive")
bayes_output = widgets.Output()


def update_bayes(*_):
    with bayes_output:
        clear_output(wait=True)

        prior = bayes_prior.value
        sensitivity = bayes_sens.value
        false_positive = bayes_fp.value

        evidence = (
            sensitivity * prior
            + false_positive * (1 - prior)
        )

        if evidence == 0:
            display(Markdown("**The evidence has probability zero.**"))
            return

        posterior = sensitivity * prior / evidence

        display(Math(
            rf"P(G)=({sensitivity:.3f})({prior:.3f})"
            rf"+({false_positive:.3f})(1-{prior:.3f})"
            rf"={evidence:.6f}"
        ))
        display(Math(
            rf"P(F\mid G)=\frac{{({sensitivity:.3f})({prior:.3f})}}{{{evidence:.6f}}}"
            rf"={posterior:.6f}"
        ))

        fig, ax = plt.subplots(figsize=(6.5, 3))
        ax.bar(["prior", "posterior"], [prior, posterior])
        ax.set_ylim(0, max(prior, posterior, 0.05) * 1.25)
        ax.set_ylabel("probability")
        ax.set_title("Bayesian updating after a positive flag")
        plt.show()


for control in (bayes_prior, bayes_sens, bayes_fp):
    control.observe(update_bayes, names="value")

display(widgets.VBox([
    bayes_prior,
    widgets.HBox([bayes_sens, bayes_fp]),
    bayes_output,
]))
update_bayes()


## 7. Historical problem: Monty Hall

Protocol:

1. the prize is uniformly placed behind one of three doors;
2. the contestant initially chooses Door 1;
3. the host knows the prize location;
4. the host always opens an unchosen goat door;
5. the host always offers a switch;
6. if two goat doors are available, the host chooses between them uniformly.

Under this protocol,

$$
P(\text{stay wins})=\frac13,
$$

and

$$
P(\text{switch wins})=\frac23.
$$

The host protocol is part of the probability model.


In [ ]:
def simulate_standard_monty(n=100_000, seed=2026):
    rng = Random(seed)
    stay_wins = 0
    switch_wins = 0

    for _ in range(n):
        car = rng.randrange(3)
        choice = 0

        host_options = [
            door
            for door in range(3)
            if door != choice and door != car
        ]
        opened = rng.choice(host_options)

        switched = next(
            door
            for door in range(3)
            if door != choice and door != opened
        )

        stay_wins += int(choice == car)
        switch_wins += int(switched == car)

    return stay_wins / n, switch_wins / n


monty_n = widgets.IntSlider(value=50_000, min=1_000, max=200_000, step=1_000, description="simulations")
monty_seed = widgets.IntSlider(value=2026, min=0, max=5000, description="seed")
monty_output = widgets.Output()


def update_monty(*_):
    with monty_output:
        clear_output(wait=True)

        stay, switch = simulate_standard_monty(
            n=monty_n.value,
            seed=monty_seed.value,
        )

        display(Math(
            rf"\widehat P(\mathrm{{stay\ win}})={stay:.4f}"
        ))
        display(Math(
            rf"\widehat P(\mathrm{{switch\ win}})={switch:.4f}"
        ))
        display(Math(
            r"P(\mathrm{stay\ win})=\frac13,\qquad"
            r"P(\mathrm{switch\ win})=\frac23"
        ))


for control in (monty_n, monty_seed):
    control.observe(update_monty, names="value")

display(widgets.VBox([
    widgets.HBox([monty_n, monty_seed]),
    monty_output,
]))
update_monty()


### A different host protocol

Changing the host's rule can change the conditional probabilities.

For example, imagine a host who chooses one of the two unchosen doors uniformly **without knowing where the prize is**, opens it, and the game continues only if a goat is revealed.

Now the information “the host opened a goat door” is generated by a different mechanism. Therefore the posterior probabilities must be recomputed from that protocol rather than copied from the standard Monty Hall model.


In [ ]:
def simulate_uninformed_host(n=200_000, seed=2026):
    rng = Random(seed)
    continued = 0
    stay_wins = 0
    switch_wins = 0

    for _ in range(n):
        car = rng.randrange(3)
        choice = 0
        other_doors = [1, 2]
        opened = rng.choice(other_doors)

        # If the host accidentally reveals the car, no switching decision follows.
        if opened == car:
            continue

        continued += 1
        switched = next(
            door for door in range(3)
            if door != choice and door != opened
        )

        stay_wins += int(choice == car)
        switch_wins += int(switched == car)

    return (
        continued / n,
        stay_wins / continued,
        switch_wins / continued,
    )


cont_rate, stay_cond, switch_cond = simulate_uninformed_host()

display(Math(
    rf"P(\mathrm{{game\ continues}})\approx {cont_rate:.4f}"
))
display(Math(
    rf"P(\mathrm{{stay\ wins}}\mid\mathrm{{continued}})\approx {stay_cond:.4f}"
))
display(Math(
    rf"P(\mathrm{{switch\ wins}}\mid\mathrm{{continued}})\approx {switch_cond:.4f}"
))
display(Markdown(
    "Under this different protocol, conditioning on continuation changes the model. "
    "This is why the host rule must be stated explicitly."
))


## 8. Independence

Events $A$ and $B$ are independent when

$$
P(A\cap B)=P(A)P(B).
$$

If $P(B)>0$, this is equivalent to

$$
P(A\mid B)=P(A).
$$

If $P(A)>0$ as well, it is also equivalent to

$$
P(B\mid A)=P(B).
$$

Independence is a property of the events **and of the probability measure**.


In [ ]:
ind_A = widgets.Text(value="2,4,6", description="A")
ind_B = widgets.Text(value="1,2,3,4", description="B")
ind_output = widgets.Output()


def update_independence(*_):
    with ind_output:
        clear_output(wait=True)

        try:
            A = parse_set(ind_A.value)
            B = parse_set(ind_B.value)
        except ValueError:
            display(Markdown("**Use comma-separated integers.**"))
            return

        if not A <= die_omega or not B <= die_omega:
            display(Markdown("**A and B must be die events.**"))
            return

        PA = probability(A, die_mass)
        PB = probability(B, die_mass)
        PAB = probability(A & B, die_mass)

        display(Math(r"P(A)=" + fmt_fraction(PA)))
        display(Math(r"P(B)=" + fmt_fraction(PB)))
        display(Math(r"P(A\cap B)=" + fmt_fraction(PAB)))
        display(Math(r"P(A)P(B)=" + fmt_fraction(PA * PB)))

        independent = PAB == PA * PB
        display(Markdown(f"**Independent:** {independent}"))

        if PB > 0:
            display(Math(
                r"P(A\mid B)="
                + fmt_fraction(conditional_probability(A, B, die_mass))
            ))


for control in (ind_A, ind_B):
    control.observe(update_independence, names="value")

display(widgets.VBox([
    widgets.HBox([ind_A, ind_B]),
    ind_output,
]))
update_independence()


## 9. Disjointness is not independence

Disjointness is the set identity

$$
A\cap B=\varnothing.
$$

Independence is the probability identity

$$
P(A\cap B)=P(A)P(B).
$$

If $A$ and $B$ are disjoint and both have positive probability, then

$$
0=P(A\cap B)\ne P(A)P(B),
$$

so they are dependent.

By contrast, independent positive-probability events generally overlap.


In [ ]:
# Disjoint but dependent
A1 = frozenset({1})
B1 = frozenset({2})

# Independent but not disjoint
A2 = frozenset({1, 2})
B2 = frozenset({1, 3, 5})

display(Markdown("### Disjoint but dependent"))
display(Math(r"A=" + fmt_set(A1) + r",\qquad B=" + fmt_set(B1)))
display(Math(r"P(A\cap B)=0"))
display(Math(
    r"P(A)P(B)=" + fmt_fraction(probability(A1, die_mass) * probability(B1, die_mass))
))

display(Markdown("### Independent but not disjoint"))
display(Math(r"A=" + fmt_set(A2) + r",\qquad B=" + fmt_set(B2)))
display(Math(r"A\cap B=" + fmt_set(A2 & B2)))
display(Math(
    r"P(A\cap B)=" + fmt_fraction(probability(A2 & B2, die_mass))
))
display(Math(
    r"P(A)P(B)=" + fmt_fraction(probability(A2, die_mass) * probability(B2, die_mass))
))


## 10. Independence is preserved by complements

If $A$ and $B$ are independent, then so are

$$
(A^c,B),\qquad
(A,B^c),\qquad
(A^c,B^c).
$$

The result follows by decomposing, for example,

$$
B=(A\cap B)\,\dot\cup\,(A^c\cap B).
$$


In [ ]:
coin_omega = frozenset({"HH", "HT", "TH", "TT"})
coin_mass = {w: Fraction(1, 4) for w in coin_omega}

A = frozenset({"HH", "HT"})
B = frozenset({"HH", "TH"})
Ac = coin_omega - A
Bc = coin_omega - B

pairs = [
    ("A,B", A, B),
    ("Aᶜ,B", Ac, B),
    ("A,Bᶜ", A, Bc),
    ("Aᶜ,Bᶜ", Ac, Bc),
]

for label, E, F in pairs:
    display(Markdown(
        f"**{label}: independent = {is_independent(E, F, coin_mass)}**"
    ))


## 11. Pairwise versus mutual independence

For events $A_1,\ldots,A_n$:

- **pairwise independence** checks every pair;
- **mutual independence** checks every finite subcollection of size at least two.

For three events, pairwise independence does not imply

$$
P(A_1\cap A_2\cap A_3)
=
P(A_1)P(A_2)P(A_3).
$$


### Pairwise independent but not mutually independent

Let

$$
\Omega=\{HH,HT,TH,TT\}
$$

with equal probabilities, and define

$$
A=\{HH,HT\},
$$

$$
B=\{HH,TH\},
$$

$$
C=\{HH,TT\}.
$$

Each event has probability $1/2$, every pairwise intersection has probability $1/4$, but

$$
P(A\cap B\cap C)=\frac14\ne\frac18.
$$


In [ ]:
A = frozenset({"HH", "HT"})
B = frozenset({"HH", "TH"})
C = frozenset({"HH", "TT"})

events_ABC = [A, B, C]
labels = ["A", "B", "C"]

for i in range(3):
    for j in range(i + 1, 3):
        E, F = events_ABC[i], events_ABC[j]
        display(Math(
            rf"P({labels[i]}\cap {labels[j]})="
            + fmt_fraction(probability(E & F, coin_mass))
            + "="
            + fmt_fraction(probability(E, coin_mass) * probability(F, coin_mass))
        ))

triple = probability(A & B & C, coin_mass)
product_marginals = (
    probability(A, coin_mass)
    * probability(B, coin_mass)
    * probability(C, coin_mass)
)

display(Math(
    r"P(A\cap B\cap C)="
    + fmt_fraction(triple)
    + r"\ne"
    + fmt_fraction(product_marginals)
))


## 12. Complement-pattern characterization of mutual independence

Events $A_1,\ldots,A_n$ are mutually independent if and only if every prescribed occurrence/non-occurrence pattern factors:

$$
P\left(\bigcap_{i=1}^{n}B_i\right)
=
\prod_{i=1}^{n}P(B_i),
$$

where for each $i$,

$$
B_i\in\{A_i,A_i^c\}.
$$

This gives a symmetric way to describe mutual independence.


In [ ]:
three_tosses = set(product((0, 1), repeat=3))
head_events = [
    {outcome for outcome in three_tosses if outcome[i] == 1}
    for i in range(3)
]


def uniform_set_prob(A, omega):
    return Fraction(len(A), len(omega))


pattern_rows = []

for pattern in product((False, True), repeat=3):
    selected = [
        event if occurs else three_tosses - event
        for event, occurs in zip(head_events, pattern)
    ]

    intersection = set(three_tosses)
    product_probability = Fraction(1, 1)

    for event in selected:
        intersection &= event
        product_probability *= uniform_set_prob(event, three_tosses)

    actual = uniform_set_prob(intersection, three_tosses)
    assert actual == product_probability

    pattern_rows.append((pattern, actual))

display(Markdown("All **8 complement patterns** factor exactly for three independent fair tosses."))

for pattern, p in pattern_rows:
    label = "".join("H" if x else "T" for x in pattern)
    display(Math(rf"P({label})=" + fmt_fraction(p)))


### Why the full-intersection identity alone is not enough

For $n\ge3$, the single equality

$$
P(A_1\cap\cdots\cap A_n)
=
\prod_{i=1}^{n}P(A_i)
$$

does not guarantee mutual independence. Other subcollections and complement patterns can fail.


In [ ]:
omega8 = frozenset(range(1, 9))
mass8 = {w: Fraction(1, 8) for w in omega8}

A8 = frozenset({1,2,3,4})
B8 = frozenset({1,2,3,5})
C8 = frozenset({1,4,5,6})

display(Math(
    r"P(A\cap B\cap C)="
    + fmt_fraction(probability(A8 & B8 & C8, mass8))
))
display(Math(
    r"P(A)P(B)P(C)="
    + fmt_fraction(
        probability(A8, mass8)
        * probability(B8, mass8)
        * probability(C8, mass8)
    )
))

pattern_prob = probability(A8 & B8 & (omega8 - C8), mass8)
pattern_product = (
    probability(A8, mass8)
    * probability(B8, mass8)
    * probability(omega8 - C8, mass8)
)

display(Math(
    r"P(A\cap B\cap C^c)="
    + fmt_fraction(pattern_prob)
    + r"\ne"
    + fmt_fraction(pattern_product)
))


## 13. At least one of mutually independent events

If $A_1,\ldots,A_n$ are mutually independent, then

$$
P\left(\bigcap_{i=1}^{n}A_i^c\right)
=
\prod_{i=1}^{n}\bigl(1-P(A_i)\bigr),
$$

and therefore

$$
P\left(\bigcup_{i=1}^{n}A_i\right)
=
1-
\prod_{i=1}^{n}\bigl(1-P(A_i)\bigr).
$$

This is an exact formula under mutual independence.

Boole's inequality,

$$
P\left(\bigcup_iA_i\right)\le\sum_iP(A_i),
$$

remains valid without independence.


In [ ]:
rare_n = widgets.IntSlider(value=100, min=1, max=1000, description="n")
rare_ind_p = widgets.FloatSlider(value=0.001, min=0.0001, max=0.05, step=0.0001, description="p")
rare_ind_output = widgets.Output()


def update_rare_independent(*_):
    with rare_ind_output:
        clear_output(wait=True)

        n = rare_n.value
        p = rare_ind_p.value

        exact = 1 - (1 - p) ** n
        union_bound = min(1.0, n * p)

        display(Math(
            rf"P(\mathrm{{at\ least\ one}})=1-(1-{p:.4f})^{{{n}}}"
            rf"={exact:.6f}"
        ))
        display(Math(
            rf"\mathrm{{Boole\ bound}}=\min\{{1,{n}({p:.4f})\}}"
            rf"={union_bound:.6f}"
        ))


for control in (rare_n, rare_ind_p):
    control.observe(update_rare_independent, names="value")

display(widgets.VBox([
    widgets.HBox([rare_n, rare_ind_p]),
    rare_ind_output,
]))
update_rare_independent()


## 14. Conditional independence

Events $A$ and $B$ are conditionally independent given $C$ when

$$
P(A\cap B\mid C)
=
P(A\mid C)P(B\mid C).
$$

Conditional independence does not generally imply unconditional independence.

Unconditional independence also need not survive conditioning.


### A common latent environment

Let $S$ be a rare high-risk regime with

$$
P(S)=0.05.
$$

Within either regime, suppose $A_1$ and $A_2$ are conditionally independent, with

$$
P(A_i\mid S)=0.20,
$$

and

$$
P(A_i\mid S^c)=0.01.
$$

Then

$$
P(A_i)
=
(0.05)(0.20)+(0.95)(0.01)
=
0.0195.
$$

But

$$
P(A_1\cap A_2)
=
(0.05)(0.20)^2+(0.95)(0.01)^2
=
0.002095,
$$

whereas

$$
P(A_1)P(A_2)
=
0.00038025.
$$

The common regime creates unconditional positive dependence.


In [ ]:
latent_pS = widgets.FloatSlider(value=0.05, min=0.001, max=0.50, step=0.001, description="P(S)")
latent_high = widgets.FloatSlider(value=0.20, min=0.001, max=0.80, step=0.001, description="P(Aᵢ|S)")
latent_low = widgets.FloatSlider(value=0.01, min=0.001, max=0.30, step=0.001, description="P(Aᵢ|Sᶜ)")
latent_output = widgets.Output()


def update_latent(*_):
    with latent_output:
        clear_output(wait=True)

        s = latent_pS.value
        ph = latent_high.value
        pl = latent_low.value

        marginal = s * ph + (1 - s) * pl
        joint = s * ph**2 + (1 - s) * pl**2
        independent_product = marginal**2
        gap = joint - independent_product

        display(Math(
            rf"P(A_i)={marginal:.6f}"
        ))
        display(Math(
            rf"P(A_1\cap A_2)={joint:.6f}"
        ))
        display(Math(
            rf"P(A_1)P(A_2)={independent_product:.6f}"
        ))
        display(Math(
            rf"P(A_1\cap A_2)-P(A_1)P(A_2)={gap:.6f}"
        ))

        if abs(gap) < 1e-12:
            display(Markdown("The selected parameters produce no unconditional dependence."))
        elif gap > 0:
            display(Markdown("The selected parameters produce **positive unconditional dependence**."))
        else:
            display(Markdown("The selected parameters produce negative unconditional dependence."))


for control in (latent_pS, latent_high, latent_low):
    control.observe(update_latent, names="value")

display(widgets.VBox([
    latent_pS,
    widgets.HBox([latent_high, latent_low]),
    latent_output,
]))
update_latent()


### Unconditional independence can fail after conditioning

Toss two fair coins and let:

- $A$: first toss is heads;
- $B$: second toss is heads;
- $C=\{HH,TT\}$.

Unconditionally, $A$ and $B$ are independent. But conditioned on $C$,

$$
P(A\cap B\mid C)=\frac12,
$$

while

$$
P(A\mid C)P(B\mid C)=\frac14.
$$

So $A$ and $B$ are not conditionally independent given $C$.


In [ ]:
C_equal = frozenset({"HH", "TT"})

cond_joint = conditional_probability(A & B, C_equal, coin_mass)
cond_A = conditional_probability(A, C_equal, coin_mass)
cond_B = conditional_probability(B, C_equal, coin_mass)

display(Math(
    r"P(A\cap B\mid C)=" + fmt_fraction(cond_joint)
))
display(Math(
    r"P(A\mid C)P(B\mid C)=" + fmt_fraction(cond_A * cond_B)
))


## 15. Two conditionally independent diagnostic tests

Suppose a disease has prevalence $0.10$. Each of two tests has sensitivity $0.90$ and specificity $0.90$, and the two test outcomes are conditionally independent given disease status.

For two positive results,

$$
P(++\mid D)=0.9^2=0.81,
$$

and

$$
P(++\mid D^c)=0.1^2=0.01.
$$

Bayes' theorem gives

$$
P(D\mid ++)=0.90.
$$

For exactly one positive result,

$$
P(\text{one +}\mid D)
=
2(0.9)(0.1)
=
0.18,
$$

and the same likelihood holds under $D^c$. Therefore the posterior remains the prior:

$$
P(D\mid\text{exactly one +})=0.10.
$$


In [ ]:
prev = Fraction(1, 10)
sens = Fraction(9, 10)
fpr = Fraction(1, 10)

p_two_pos_D = sens**2
p_two_pos_notD = fpr**2

posterior_two = (
    prev * p_two_pos_D
    / (
        prev * p_two_pos_D
        + (1 - prev) * p_two_pos_notD
    )
)

p_one_D = 2 * sens * (1 - sens)
p_one_notD = 2 * fpr * (1 - fpr)

posterior_one = (
    prev * p_one_D
    / (
        prev * p_one_D
        + (1 - prev) * p_one_notD
    )
)

display(Math(
    r"P(D\mid ++)=" + fmt_fraction(posterior_two)
))
display(Math(
    r"P(D\mid\mathrm{exactly\ one\ +})="
    + fmt_fraction(posterior_one)
))


## 16. Guided exercise generator

The exercises mix calculation with structural interpretation.


In [ ]:
exercise_rng = Random(20260815)

exercise_type = widgets.Dropdown(
    options=[
        ("Random", "random"),
        ("Conditional probability", "conditional"),
        ("Chain rule", "chain"),
        ("Total probability", "total"),
        ("Bayes", "bayes"),
        ("Independence", "independence"),
        ("Pairwise vs mutual", "pairwise"),
        ("At least one independent event", "atleast"),
    ],
    value="random",
    description="Type",
)

new_exercise = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
check_button = widgets.Button(description="Check")
answer_box = widgets.Text(description="Answer")
exercise_prompt = widgets.Output()
exercise_feedback = widgets.Output()
exercise_state = {}


def make_exercise(_=None):
    kind = exercise_type.value
    if kind == "random":
        kind = exercise_rng.choice([
            "conditional",
            "chain",
            "total",
            "bayes",
            "independence",
            "pairwise",
            "atleast",
        ])

    if kind == "conditional":
        answer = 0.25
        prompt = "If E⊆T, P(E)=0.05 and P(T)=0.20, find P(E|T)."
        hint = "Because E⊆T, E∩T=E."
        solution = r"P(E\mid T)=0.05/0.20=0.25."

    elif kind == "chain":
        answer = 0.008
        prompt = (
            "If P(A₁)=0.08, P(A₂|A₁)=0.25 and "
            "P(A₃|A₁∩A₂)=0.40, find P(A₁∩A₂∩A₃)."
        )
        hint = "Multiply the three chain-rule factors."
        solution = r"(0.08)(0.25)(0.40)=0.008."

    elif kind == "total":
        answer = 0.018
        prompt = (
            "Two production lines make 60% and 40% of items. "
            "Their defect rates are 1% and 3%. Find P(defect)."
        )
        hint = "Average the conditional defect rates using the line shares."
        solution = r"P(D)=(0.01)(0.60)+(0.03)(0.40)=0.018."

    elif kind == "bayes":
        answer = 0.5625
        prompt = (
            "Source R has prior 0.70 and source L prior 0.30. "
            "P(A|R)=0.04 and P(A|L)=0.12. Find P(L|A)."
        )
        hint = "First compute P(A) by total probability."
        solution = r"P(A)=0.064,\qquad P(L\mid A)=0.036/0.064=0.5625."

    elif kind == "independence":
        answer = "yes"
        prompt = (
            "If P(A)=0.10, P(B)=0.20 and P(A∩B)=0.02, "
            "are A and B independent? Answer yes or no."
        )
        hint = "Compare P(A∩B) with P(A)P(B)."
        solution = r"0.02=(0.10)(0.20),\text{ so yes.}"

    elif kind == "pairwise":
        answer = "no"
        prompt = (
            "If three events are pairwise independent, must they be mutually "
            "independent? Answer yes or no."
        )
        hint = "Recall the two-coin example A, B, C."
        solution = r"\text{No. Pairwise independence does not control the triple intersection.}"

    else:
        n = 100
        p = 0.001
        answer = 1 - (1 - p) ** n
        prompt = (
            "100 mutually independent rare events each have probability 0.001. "
            "Find the probability that at least one occurs."
        )
        hint = "Compute one minus the probability that none occur."
        solution = r"1-(0.999)^{100}\approx0.09521."

    exercise_state.clear()
    exercise_state.update(
        answer=str(answer).lower(),
        hint=hint,
        solution=solution,
    )
    answer_box.value = ""

    with exercise_prompt:
        clear_output(wait=True)
        display(Markdown("### Exercise\n" + prompt))

    with exercise_feedback:
        clear_output(wait=True)


def show_hint(_):
    with exercise_feedback:
        clear_output(wait=True)
        display(Markdown("**Hint:** " + exercise_state["hint"]))


def reveal_solution(_):
    with exercise_feedback:
        clear_output(wait=True)
        display(Math(exercise_state["solution"]))


def check_answer(_):
    with exercise_feedback:
        clear_output(wait=True)

        guess = answer_box.value.strip().lower().replace(" ", "")
        target = exercise_state["answer"].replace(" ", "")

        try:
            if target not in {"yes", "no"}:
                if abs(float(guess) - float(target)) < 5e-5:
                    display(Markdown("**Correct.**"))
                    return
        except Exception:
            pass

        if guess == target:
            display(Markdown("**Correct.**"))
        else:
            display(Markdown(
                "**Not yet.** Identify the model and theorem before computing."
            ))


new_exercise.on_click(make_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal_solution)
check_button.on_click(check_answer)

display(widgets.VBox([
    widgets.HBox([exercise_type, new_exercise]),
    exercise_prompt,
    widgets.HBox([answer_box, check_button]),
    widgets.HBox([hint_button, reveal_button]),
    exercise_feedback,
]))

make_exercise()


## 17. AI Audit

Conditional probability is especially vulnerable to modelling errors because a short formula can hide an incorrectly specified information structure.

### Audit protocol

For any AI-generated solution, check:

1. Is the conditioning event explicitly stated?
2. Is its probability positive?
3. Has the AI reversed $P(A\mid B)$ and $P(B\mid A)$?
4. If the multiplication rule is used, is the conditional factor conditioned on the correct previous events?
5. Does a proposed partition consist of mutually exclusive and exhaustive events?
6. In Bayes' theorem, are prior, likelihood and posterior kept distinct?
7. Does the denominator equal the total probability of the evidence?
8. In Monty Hall, is the host protocol fully specified?
9. Is disjointness being confused with independence?
10. Is pairwise independence being mistaken for mutual independence?
11. Is one full-intersection equality being mistaken for mutual independence?
12. If an exact union formula is used, has mutual independence actually been assumed?
13. Is conditional independence being incorrectly promoted to unconditional independence?
14. Could a common latent cause explain apparent dependence?

### Suggested AI audit prompts

- “Explain why conditioning on a fixed event produces a probability measure.”
- “Solve a Bayes problem but label prior, likelihood, evidence and posterior before substituting numbers.”
- “Give one disjoint-but-dependent example and one independent-but-overlapping example.”
- “Verify all complement patterns for three independent fair tosses.”
- “Show numerically how a latent regime can make conditionally independent events dependent unconditionally.”
- “Simulate Monty Hall under two different host protocols and explain why the answers differ.”


## 18. Self-check quiz


In [ ]:
quiz_data = [
    (
        "1. Conditional probability P(A|B) requires:",
        ["Choose...", "P(A)>0", "P(B)>0", "A⊆B", "independence"],
        "P(B)>0",
        r"P(A\mid B)=P(A\cap B)/P(B).",
    ),
    (
        "2. If A and B are independent and P(B)>0, then:",
        ["Choose...", "P(A|B)=0", "P(A|B)=P(A)", "P(A|B)=P(B)", "A∩B=∅"],
        "P(A|B)=P(A)",
        r"P(A\mid B)=P(A).",
    ),
    (
        "3. Positive-probability disjoint events are:",
        ["Choose...", "always independent", "necessarily dependent", "mutually independent", "undefined"],
        "necessarily dependent",
        r"P(A\cap B)=0\ne P(A)P(B).",
    ),
    (
        "4. Bayes' theorem updates:",
        ["Choose...", "likelihood to prior", "prior to posterior after evidence", "posterior to sample space", "independence to disjointness"],
        "prior to posterior after evidence",
        r"\text{Bayes reverses the direction from cause-to-evidence to evidence-to-cause.}",
    ),
    (
        "5. Pairwise independence implies mutual independence:",
        ["Choose...", "true", "false"],
        "false",
        r"\text{The two-toss A,B,C example is a counterexample.}",
    ),
    (
        "6. For mutually independent A_i, P(at least one) equals:",
        ["Choose...", "ΣP(A_i)", "1−∏(1−P(A_i))", "∏P(A_i)", "1−ΣP(A_i)"],
        "1−∏(1−P(A_i))",
        r"P(\cup_iA_i)=1-\prod_i(1-P(A_i)).",
    ),
    (
        "7. Conditional independence given S guarantees unconditional independence:",
        ["Choose...", "true", "false"],
        "false",
        r"\text{A latent regime can induce unconditional dependence.}",
    ),
    (
        "8. In standard Monty Hall, switching wins with probability:",
        ["Choose...", "1/3", "1/2", "2/3", "1"],
        "2/3",
        r"P(\mathrm{switch\ win})=2/3.",
    ),
    (
        "9. A likelihood P(A|B_j) must sum to 1 over j:",
        ["Choose...", "true", "false"],
        "false",
        r"\text{Likelihoods as functions of causes need not sum to one.}",
    ),
    (
        "10. The host protocol in Monty Hall is:",
        ["Choose...", "irrelevant", "part of the probability model"],
        "part of the probability model",
        r"\text{Changing the protocol can change the posterior probabilities.}",
    ),
]

quiz_widgets = []
quiz_rows = []

for prompt, options, _, _ in quiz_data:
    dropdown = widgets.Dropdown(
        options=options,
        value="Choose...",
        layout=widgets.Layout(width="350px"),
    )
    quiz_widgets.append(dropdown)
    quiz_rows.append(widgets.HBox([
        widgets.HTML(f"<div style='width:660px'>{prompt}</div>"),
        dropdown,
    ]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()


def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)

        score = sum(
            widget.value == correct
            for widget, (_, _, correct, _) in zip(
                quiz_widgets,
                quiz_data,
            )
        )

        display(Markdown(f"### Score: {score}/{len(quiz_data)}"))

        for i, (
            widget,
            (_, _, correct, explanation),
        ) in enumerate(zip(quiz_widgets, quiz_data), 1):
            mark = "✓" if widget.value == correct else "✗"
            display(Markdown(
                f"**{mark} Question {i}:** correct answer = `{correct}`"
            ))
            display(Math(explanation))


grade_button.on_click(grade_quiz)
display(widgets.VBox(
    quiz_rows + [grade_button, quiz_output]
))


## 19. Automatic mathematical verification

This cell checks exact Bayes calculations, conditional probability identities, independence examples, complement preservation, complement patterns and the rare-event formula.


In [ ]:
# Exact screening Bayes calculation.
prior_target = Fraction(2, 100)
sensitivity = Fraction(90, 100)
false_positive = Fraction(5, 100)

p_flag = (
    sensitivity * prior_target
    + false_positive * (1 - prior_target)
)
posterior_target = sensitivity * prior_target / p_flag

assert p_flag == Fraction(67, 1000)
assert posterior_target == Fraction(18, 67)


# Conditioning on even die outcomes creates a probability measure.
B_even = frozenset({2, 4, 6})
conditional_point_masses = {
    w: conditional_probability({w}, B_even, die_mass)
    for w in die_omega
}

assert sum(
    conditional_point_masses.values(),
    Fraction(0, 1),
) == 1

assert conditional_point_masses[2] == Fraction(1, 3)
assert conditional_point_masses[4] == Fraction(1, 3)
assert conditional_point_masses[6] == Fraction(1, 3)
assert conditional_point_masses[1] == 0


# Multiplication rule in the without-replacement example.
assert Fraction(3, 5) * Fraction(2, 4) == Fraction(3, 10)


# Total probability in production-line example.
assert (
    Fraction(1, 100) * Fraction(60, 100)
    + Fraction(3, 100) * Fraction(40, 100)
) == Fraction(18, 1000)


# Independence and complements for two fair tosses.
assert is_independent(A, B, coin_mass)
assert is_independent(coin_omega - A, B, coin_mass)
assert is_independent(A, coin_omega - B, coin_mass)
assert is_independent(coin_omega - A, coin_omega - B, coin_mass)


# Pairwise-but-not-mutual example.
A_pw = frozenset({"HH", "HT"})
B_pw = frozenset({"HH", "TH"})
C_pw = frozenset({"HH", "TT"})

for E, F in [(A_pw, B_pw), (A_pw, C_pw), (B_pw, C_pw)]:
    assert is_independent(E, F, coin_mass)

assert (
    probability(A_pw & B_pw & C_pw, coin_mass)
    != probability(A_pw, coin_mass)
    * probability(B_pw, coin_mass)
    * probability(C_pw, coin_mass)
)


# All 2^3 complement patterns for three fair tosses.
coin_outcomes = set(product((0, 1), repeat=3))
head_events = [
    {outcome for outcome in coin_outcomes if outcome[i] == 1}
    for i in range(3)
]

for occurrence_pattern in product((False, True), repeat=3):
    selected_events = [
        event if occurs else coin_outcomes - event
        for event, occurs in zip(head_events, occurrence_pattern)
    ]

    intersection = coin_outcomes.copy()
    product_probability = Fraction(1, 1)

    for event in selected_events:
        intersection &= event
        product_probability *= Fraction(
            len(event),
            len(coin_outcomes),
        )

    assert Fraction(
        len(intersection),
        len(coin_outcomes),
    ) == product_probability


# Exact probability of at least one rare independent event.
at_least_one = 1 - Fraction(999, 1000) ** 100

assert float(at_least_one) < 0.1
assert float(at_least_one) > 0.095


# Latent-regime example.
pS = Fraction(5, 100)
high = Fraction(20, 100)
low = Fraction(1, 100)

marginal = pS * high + (1 - pS) * low
joint = pS * high**2 + (1 - pS) * low**2

assert marginal == Fraction(195, 10000)
assert joint == Fraction(2095, 1000000)
assert joint > marginal**2


show_result(
    "All Chapter 4 automatic checks passed",
    r"P(F\mid G)=\frac{18}{67}",
    r"P(R_1\cap B_2)=\frac3{10}",
    r"P(D)=0.018",
    r"P(A\cap B)=P(A)P(B)\Longleftrightarrow P(A\mid B)=P(A)",
    r"P(\cup_iA_i)=1-\prod_i(1-P(A_i))\quad\text{under mutual independence}",
    note=(
        "Bayes, conditioning, independence, complement patterns, "
        "rare-event unions and latent-regime dependence all passed."
    ),
)


## 20. Chapter map

| Chapter concept | Computational representation |
|---|---|
| conditional probability | finite event calculator |
| conditional probability measure | renormalized die model |
| multiplication rule | two draws without replacement |
| chain rule | sequential probability widget |
| law of total probability | production-line mixture |
| Bayes' theorem | screening posterior |
| Monty Hall | exact model plus simulation |
| host-protocol dependence | alternative uninformed-host simulation |
| independence | product and conditional tests |
| disjointness vs independence | explicit counterexamples |
| complements of independent events | exact two-coin checks |
| pairwise vs mutual independence | classic three-event counterexample |
| complement-pattern criterion | all $2^3$ patterns |
| at least one independent event | exact product formula vs union bound |
| conditional independence | latent-regime model |
| repeated diagnostic tests | Bayes under conditional independence |

The guiding principle is:

> Conditional probability changes the probability measure to reflect information. Independence says that this update leaves a specified probability unchanged.
